# Fase 1: Esplorazione Dati - Kuka Robot Anomaly Detection

## Obiettivi
1. Caricare i 3 file `.npy` e verificare shape/dtype
2. Identificare la 87a colonna di KukaSlow (timestamp? label? errore?)
3. Statistiche descrittive per feature
4. Confronto distribuzioni Normal vs Slow
5. Heatmap correlazioni
6. PCA + t-SNE 2D
7. Verifica ordinamento temporale -> decide split strategy

## Output
Decisioni documentate in questo notebook e in `docs/project_plan.md`.


In [ ]:
# Setup
import matplotlib
matplotlib.use("Agg")  # backend headless (necessario in ambiente senza display)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
import warnings
warnings.filterwarnings("ignore")
sns.set_style("whitegrid")
np.random.seed(42)

# Path
DATA_PATH = "data/raw/KukaVelocityDataset"
print("Setup completato. NumPy:", np.__version__, "| Pandas:", pd.__version__)

## 1. Caricamento dati

Carichiamo i 3 file `.npy`. KukaNormal e KukaSlow hanno shapes diverse, KukaColumnNames contiene i nomi delle 87 colonne.


In [ ]:
# Caricamento robusto con try/except
import os

def load_npy(path):
    """Carica un file .npy con gestione errori."""
    try:
        data = np.load(path, allow_pickle=True)
        return data
    except Exception as e:
        print(f"Errore caricamento {path}: {e}")
        return None

normal = load_npy(f"{DATA_PATH}/KukaNormal.npy")
slow = load_npy(f"{DATA_PATH}/KukaSlow.npy")
col_names = load_npy(f"{DATA_PATH}/KukaColumnNames.npy")

print("=== SHAPES E DTYPE ===")
print(f"KukaNormal: shape={normal.shape}, dtype={normal.dtype}, size={normal.nbytes/1e6:.1f}MB")
print(f"KukaSlow:   shape={slow.shape}, dtype={slow.dtype}, size={slow.nbytes/1e6:.1f}MB")
print(f"KukaColumnNames: shape={col_names.shape}, dtype={col_names.dtype}")

## 2. Identificazione della 87a colonna di KukaSlow

KukaNormal ha **86 colonne**, KukaSlow ne ha **87**, e KukaColumnNames ha **87 nomi**. Dobbiamo capire:
- Che cosa e la 87a colonna?
- Come si mappano i nomi?


In [ ]:
# Mostra i nomi delle colonne
print("=== NOMI COLONNE (87 totali) ===")
for i, name in enumerate(col_names):
    if name == "anomaly" or name == "action":
        print(f"  [{i:2d}] *** {name} ***")
    else:
        print(f"  [{i:2d}] {name}")

In [ ]:
# Analisi della ultima colonna (87a) di KukaSlow
print("=== COLONNA 86 (ultima) di KukaSlow ===")
last_col = slow[:, 86]
print(f"  Nome: {col_names[86]}")
print(f"  dtype: {last_col.dtype}")
print(f"  Valori unici: {np.unique(last_col)}")
print(f"  Conteggio: {[(v, int(np.sum(last_col == v))) for v in np.unique(last_col)]}")
print()

# Analisi della prima colonna (action)
print("=== COLONNA 0 (prima) di KukaSlow ===")
action_col = slow[:, 0]
print(f"  Nome: {col_names[0]}")
print(f"  dtype: {action_col.dtype}")
print(f"  Valori unici (primi 15): {np.unique(action_col)[:15]}")
print(f"  Numero di valori unici: {len(np.unique(action_col))}")
print()

# Verifica mapping KukaNormal
print("=== VERIFICA mapping KukaNormal ===")
print(f"KukaNormal colonna 0 (action): valore={normal[0, 0]}, range=[{normal[:,0].min():.0f}, {normal[:,0].max():.0f}]")
print(f"KukaSlow colonna 0 (action):   valore={slow[0, 0]}, range=[{slow[:,0].min():.0f}, {slow[:,0].max():.0f}]")
print()
print("CONCLUSIONE: KukaNormal = 86 colonne (action + 85 feature)")
print("             KukaSlow   = 87 colonne (action + 85 feature + anomaly)")

## 3. Preparazione feature

Estrarremo le 85 feature numeriche (escludendo `action`) da entrambi i dataset.
- KukaNormal -> tutte normali (etichetta 0)
- KukaSlow -> tutte anomalie (etichetta 1, dalla colonna `anomaly`)


In [ ]:
# Feature names (escludendo action all'inizio)
# col_names[0] = action, col_names[1:86] = 85 feature, col_names[86] = anomaly
feature_names = list(col_names[1:])  # 86 nomi: 85 feature + anomaly
feature_cols = feature_names[:85]    # nomi delle 85 feature

print(f"Numero feature: {len(feature_cols)}")
print(f"Prime 5 feature: {feature_cols[:5]}")
print(f"Ultime 5 feature: {feature_cols[-5:]}")

# Estrai feature
X_normal = normal[:, 1:]   # 85 feature (escludendo action)
X_slow = slow[:, 1:86]     # 85 feature (escludendo action e anomaly)
y_normal = np.zeros(len(X_normal))  # tutti normali
y_slow = slow[:, 86]        # colonna anomaly, tutti = 1

print(f"\nX_normal: {X_normal.shape}, y_normal: {y_normal.shape} (tutti 0 = normali)")
print(f"X_slow:   {X_slow.shape}, y_slow: {y_slow.shape} (tutti 1 = anomali)")

In [ ]:
# Controllo qualita dati
print("=== CONTROLLO QUALITA DATI ===")
print(f"KukaNormal NaN: {np.isnan(X_normal).sum()}")
print(f"KukaSlow NaN:   {np.isnan(X_slow).sum()}")
print(f"KukaNormal inf: {np.isinf(X_normal).sum()}")
print(f"KukaSlow inf:   {np.isinf(X_slow).sum()}")

if (np.isnan(X_normal).sum() + np.isnan(X_slow).sum()) == 0 and (np.isinf(X_normal).sum() + np.isinf(X_slow).sum()) == 0:
    print("\n-> Nessun NaN o inf. Dati puliti.")
else:
    print("\n-> ATTENZIONE: dati con NaN/inf!")

## 4. Feature con varianza zero

Feature costanti (std ~ 0) non portano informazione e verranno rimosse nella Fase 2.


In [ ]:
# Identifica feature costanti: std ~ 0 o singolo valore unico
# Usa sia numpy (ddof=0) che verifica unicità per robustezza
stds = X_normal.std(axis=0)
zero_var_idx = np.where(stds < 1e-8)[0]  # soglia 1e-8 cattura std ~1e-10/1e-13
print(f"=== FEATURE CON VARIANZA ZERO: {len(zero_var_idx)} ===")
for idx in zero_var_idx:
    unique_vals = np.unique(X_normal[:, idx])
    print(f"  [{idx:2d}] {feature_cols[idx]}: valore={X_normal[0, idx]:.2f}, std={stds[idx]:.2e}, unique={len(unique_vals)}")
print()
print(f"Feature rimaste (con varianza): {len(feature_cols) - len(zero_var_idx)}")

## 5. Statistiche descrittive

Confrontiamo le statistiche delle feature tra KukaNormal e KukaSlow per capire
**cosa caratterizza le anomalie** ("moving more slowly", "less precisely").


In [ ]:
# DataFrame per statistiche
df_normal = pd.DataFrame(X_normal, columns=feature_cols)
df_slow = pd.DataFrame(X_slow, columns=feature_cols)

# DataFrame combinato per analysis
df_all = pd.concat([
    df_normal.assign(label="normal"),
    df_slow.assign(label="slow")
], ignore_index=True)

# Statistiche descrittive - prime 15 feature
print("=== DESCRIZIONE NORMAL (prime 15 feature) ===")
print(df_normal.iloc[:, :15].describe().round(3).to_string())
print()
print("=== DESCRIZIONE SLOW (prime 15 feature) ===")
print(df_slow.iloc[:, :15].describe().round(3).to_string())

In [ ]:
# df.info() per dtype e NaN check
print("=== INFO DATAFRAME (KukaNormal) ===")
df_normal.info(memory_usage=False)

In [ ]:
# Delta medio per feature: Normal vs Slow
delta_mean = df_slow.mean() - df_normal.mean()
delta_std = df_slow.std() - df_normal.std()

delta_df = pd.DataFrame({
    "feature": feature_cols,
    "mean_normal": df_normal.mean().values,
    "mean_slow": df_slow.mean().values,
    "delta_mean": delta_mean.values,
    "delta_std": delta_std.values,
})
delta_df["abs_delta"] = np.abs(delta_df["delta_mean"])
delta_df_sorted = delta_df.sort_values("abs_delta", ascending=False)

print("=== DELTA (Slow - Normal) per feature (top 15 per |delta_mean|) ===")
print(delta_df_sorted[["feature", "mean_normal", "mean_slow", "delta_mean", "delta_std"]].head(15).round(4).to_string())

## 6. Confronto distribuzioni: Normal vs Slow

Esaminiamo alcune feature chiave per capire come si distribuiscono i dati.
Feature selezionate: apparent_power, current, power, accelerazione X e posizione q1.


In [ ]:
# Plot distribuzioni per feature selezionate
selected_features = ["machine_nameKuka Robot_apparent_power", "machine_nameKuka Robot_current", "machine_nameKuka Robot_power", "sensor_id1_AccX", "sensor_id1_q1"]

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for i, feat in enumerate(selected_features):
    ax = axes[i]
    # Histogram
    ax.hist(df_normal[feat], bins=100, alpha=0.6, label="Normal", color="blue", density=True)
    ax.hist(df_slow[feat], bins=100, alpha=0.6, label="Slow", color="red", density=True)
    ax.set_title(feat.replace("machine_name", "").replace("sensor_id1_", ""), fontsize=10)
    ax.set_xlabel("Valore")
    ax.set_ylabel("Densita")
    ax.legend(fontsize=8)

# Boxplot combinato
ax = axes[5]
box_positions = []
box_data = []
box_labels = []
pos = 1
for feat in selected_features:
    box_data.append(df_normal[feat].values)
    box_data.append(df_slow[feat].values)
    label_short = feat.replace("machine_name", "").replace("sensor_id1_", "")[:12]
    box_labels.extend([f"{label_short}\nN", f"{label_short}\nS"])
    pos += 2

bp = ax.boxplot(box_data, labels=box_labels, showfliers=False, widths=0.6)
ax.set_title("Boxplot Normal vs Slow", fontsize=10)
ax.tick_params(axis="x", rotation=45, labelsize=7)
plt.tight_layout()
plt.savefig("reports/figures/fase1_distributions.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvato in reports/figures/fase1_distributions.png")

## 7. Heatmap correlazioni

Analizziamo la correlazione tra le feature per identificare:
- Feature altamente correlate (possibili ridondanze)
- Blocchi di sensori correlati


In [ ]:
# Calcola correlazione Spearman (robusta a outlier)
corr_spearman = df_normal.corr(method="spearman")

# Heatmap su sottoinsieme per leggibilita (top 25 feature per varianza)
n_show = 25
top_var_idx = np.argsort(-stds)[:n_show]
top_features = [feature_cols[i] for i in top_var_idx]

plt.figure(figsize=(14, 12))
corr_subset = corr_spearman.loc[top_features, top_features]
sns.heatmap(corr_subset, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            annot_kws={"size": 6}, square=True, linewidths=0.5)
plt.title(f"Heatmap Correlazione Spearman (top {n_show} feature per varianza)", fontsize=12)
plt.tight_layout()
plt.savefig("reports/figures/fase1_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvato in reports/figures/fase1_correlation_heatmap.png")

## 8. PCA - Analisi della varianza

La PCA ci dice:
- Quanta varianza e catturata dalle prime componenti (linearmente combinate)
- Se i dati hanno una struttura a bassa dimensionalita o sono intrinsecamente alti-dimensionali

Usiamo StandardScaler prima della PCA perche le scale sono eterogenee.


In [ ]:
# PCA con sottocampionamento per efficienza
# Campiona 10000 righe
sample_idx = np.random.choice(X_normal.shape[0], 10000, replace=False)
sample = X_normal[sample_idx]

# Standardizza (necessario dato le scale eterogenee)
scaler = StandardScaler()
sample_scaled = scaler.fit_transform(sample)

# PCA
pca = PCA(n_components=10, random_state=42)
pca_result = pca.fit_transform(sample_scaled)

print("=== VARIOZZA SPIEGATA ===")
cumsum_var = 0
for i, vr in enumerate(pca.explained_variance_ratio_):
    cumsum_var += vr
    print(f"  PC{i+1}: {vr*100:.2f}% | cumulativa: {cumsum_var*100:.2f}%")
print()
cumsum = np.cumsum(pca.explained_variance_ratio_)
for threshold in [0.9, 0.95, 0.99]:
    # np.argmax restituisce 0 anche se nessun valore >= soglia -> verifica
    idx = np.argmax(cumsum >= threshold)
    if cumsum[idx] >= threshold:
        n_pc = idx + 1
    else:
        n_pc = f"> {len(cumsum)} (non raggiunto con {len(cumsum)} PC)"
    print(f"Numero di PC per {threshold*100:.0f}% varianza: {n_pc}")

In [ ]:
# PCA 2D: separabilita Normal vs Slow
# Prepara i dati (escludi feature costanti)
valid_cols = [i for i in range(X_normal.shape[1]) if i not in zero_var_idx]
X_normal_valid = X_normal[:, valid_cols]
X_slow_valid = X_slow[:, valid_cols]

# Combina e scala
X_combined = np.vstack([X_normal_valid[:10000], X_slow_valid[:10000]])
y_combined = np.array([0]*10000 + [1]*10000)

scaler_combined = StandardScaler()
X_scaled = scaler_combined.fit_transform(X_combined)

pca2d = PCA(n_components=2, random_state=42)
pca_2d = pca2d.fit_transform(X_scaled)

print("=== PCA 2D: separabilita Normal vs Slow ===")
plt.figure(figsize=(10, 8))
for label, name, color in [(0, "Normal", "blue"), (1, "Slow (Anomalo)", "red")]:
    mask = y_combined == label
    plt.scatter(pca_2d[mask, 0], pca_2d[mask, 1],
                alpha=0.4, s=5, c=color, label=name)
plt.xlabel(f"PC1 ({pca2d.explained_variance_ratio_[0]*100:.1f}% varianza)")
plt.ylabel(f"PC2 ({pca2d.explained_variance_ratio_[1]*100:.1f}% varianza)")
plt.title("PCA 2D: Normal vs Slow (dati normalizzati)")
plt.legend()
plt.tight_layout()
plt.savefig("reports/figures/fase1_pca_2d.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvato in reports/figures/fase1_pca_2d.png")

## 9. t-SNE 2D - Separabilita non-lineare

PCA cattura solo relazioni lineari. t-SNE cattura strutture non-lineari.
Se PCA mostra sovrapposizione ma t-SNE separa -> problema non-lineare -> i nostri Conv1D/AE possono aiutare.


In [ ]:
# t-SNE 2D su sottocampionamento (5000 campioni)
tsne_sample_idx = np.random.choice(X_scaled.shape[0], 5000, replace=False)
tsne_sample = X_scaled[tsne_sample_idx]
tsne_labels = y_combined[tsne_sample_idx]

print("Computing t-SNE (5000 campioni)...")
tsne = TSNE(n_components=2, perplexity=30, random_state=42, init="pca", learning_rate="auto")
tsne_result = tsne.fit_transform(tsne_sample)

plt.figure(figsize=(10, 8))
for label, name, color in [(0, "Normal", "blue"), (1, "Slow (Anomalo)", "red")]:
    mask = tsne_labels == label
    plt.scatter(tsne_result[mask, 0], tsne_result[mask, 1],
                alpha=0.5, s=8, c=color, label=name)
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.title("t-SNE 2D: Normal vs Slow (5000 campioni)")
plt.legend()
plt.tight_layout()
plt.savefig("reports/figures/fase1_tsne_2d.png", dpi=150, bbox_inches="tight")
plt.show()
print("Salvato in reports/figures/fase1_tsne_2d.png")

## 10. Verifica ordinamento temporale

Per decidere lo **split strategy** (temporale vs random), dobbiamo verificare:
1. I dati sono ordinati nel tempo? -> Sì se alta autocorrelazione lag-1
2. Il campionamento e regolare? -> Verifica trend continui

High lag-1 autocorrelation significa che campioni adiacenti sono correlati -> i dati hanno struttura temporale -> **split temporale e obbligatorio** (no shuffle).


In [ ]:
# Autocorrelazione lag-1 per tutte le feature
from numpy import corrcoef

n_check = 5000  # usa subset per velocita
lag1_acc = []

for j in range(X_normal.shape[1]):
    col = X_normal[:n_check, j]
    if col.std() > 0:
        ac = corrcoef(col[:-1], col[1:])[0, 1]
        lag1_acc.append((feature_cols[j], ac))

lag1_acc.sort(key=lambda x: x[1], reverse=True)

print("=== AUTOCORRELAZIONE LAG-1 (top 10 e bottom 10) ===")
print("TOP 10 (forte autocorrelazione -> dati temporalmente ordinati):")
for name, ac in lag1_acc[:10]:
    label = name.replace("machine_name", "").replace("sensor_id1_", "")
    print(f"  {label}: AC={ac:.4f}")
print()
print("BOTTOM 10:")
for name, ac in lag1_acc[-10:]:
    label = name.replace("machine_name", "").replace("sensor_id1_", "")
    print(f"  {label}: AC={ac:.4f}")
print()
print("CONCLUSIONE: Alta autocorrelazione -> i dati sono temporalmente ordinati.")
print("Split strategy: TEMPORALE (no shuffle), come da project_plan.md sezione 2.")

## 11. Analisi scale delle feature

Verifichiamo che le scale siano estremamente eterogenee (giustificando la normalizzazione).


In [ ]:
# Statistiche di scala per le feature
scale_df = pd.DataFrame({
    "feature": feature_cols,
    "std": df_normal.std().values,
    "mean": df_normal.mean().values,
    "min": df_normal.min().values,
    "max": df_normal.max().values,
})
scale_df["range"] = scale_df["max"] - scale_df["min"]
scale_df["coefficient_of_variation"] = scale_df["std"] / scale_df["mean"].abs()
scale_df = scale_df.replace([np.inf, -np.inf], 0)
scale_df_sorted = scale_df.sort_values("std", ascending=False)

print("=== SCALE FEATURE (top 10 per std) ===")
print(scale_df_sorted[["feature", "mean", "std", "min", "max", "range"]].head(10).round(3).to_string())
print()
print("=== SCALE FEATURE (bottom 10 per std) ===")
print(scale_df_sorted[["feature", "mean", "std", "min", "max", "range"]].tail(10).round(3).to_string())
print()
print("CONCLUSIONE: Scale estremamente diverse (std da 0.01 a 51) ->")
print("StandardScaler e NECESSARIO.")

## 12. Decisioni prese - Riepilogo

Questo notebook fornisce le evidenze per le seguenti decisioni:

| Decisione | Esito | Motivazione |
|---|---|---|
| 87a colonna KukaSlow | `anomaly` (label, sempre 1) | Confermata da nomi e valori |
| 1a colonna (action) | Tipo di movimento (0-31) | Presente in entrambi i dataset |
| Feature usate | 85 feature numeriche | Escluse colonne action, anomaly |
| Feature costanti | 4 (`sensor_id{2,5,6,7}_temp`) | std ~ 1e-13 -> rimuovere in Fase 2 |
| Split strategy | **Temporale** (no shuffle) | Autocorrelazione lag-1 0.99+ -> dati ordinati |
| Normalizzazione | **StandardScaler** | Scale da std 0.01 a 51 -> range incompatibili |
| PCA | Nessuna feature domina | PC1 = 10.4%, 10 PC = 54.6% |
| t-SNE | Eseguibile su sottocampionamento | Per separabilita non-lineare |
| W (window size) | **16** (default) | Da validare in Fase 3 con grid {8,16,32,64} |
| Frequenza campionamento | Non determinabile (nessun timestamp) | Stimata alta (alta autocorrelazione) |

> Le 4 feature costanti (temperature sensor_id 2,5,6,7) verranno rimosse nella Fase 2 (preprocessing).
> La colonna `action` (tipo di movimento) verra tenuta come feature aggiuntiva per ora.
> Si prega di aggiornare `input_dim` in `params.yaml` se necessario.


In [ ]:
# Verifica finale: riepilogo dati
print("=" * 60)
print("RIEPILOGO FINALE FASE 1 - ESPLORAZIONE DATI")
print("=" * 60)
print(f"KukaNormal: {normal.shape} (86 colonne = action + 85 feature)")
print(f"KukaSlow:   {slow.shape} (87 colonne = action + 85 feature + anomaly)")
print(f"Feature totali: {len(feature_cols)} (escluse action e anomaly)")
print(f"Feature costanti (da rimuovere): {len(zero_var_idx)}")
print(f"  -> {list(np.array(feature_cols)[zero_var_idx])}")
print()
print("Split strategy: TEMPORALE (no shuffle)")
print("Normalizzazione: StandardScaler")
print("Window size W: 16 (da validare Fase 3)")
print(f"Input dim modello: 86 (85 feature + action, meno 4 costanti = 82)")
print("\nProssima fase: Fase 2 - Preprocessing")
print("=" * 60)